In [15]:
import csv
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# Text to Vectors
  - Why Convert Words to Vectors?
  Machine learning models can only understand numbers, not raw text. To train models for
tasks like classification, translation, sentiment analysis, etc., we need to convert words or
documents into fixed-size vectors.

In [16]:
df = pd.DataFrame({'text': ['he cooks food','food cooks good', \
                            'food smells good', 'food is food'], \
                   'output':[0,1,1,0]})
print(df)

               text  output
0     he cooks food       0
1   food cooks good       1
2  food smells good       1
3      food is food       0


## 1. One Hot Encoding
- One-Hot Encoding represents each word in a vocabulary as a binary vector:
  - Each word is represented by a vector the same length as the vocabulary.

  - The position corresponding to that word is 1, and all other positions are 0.

  - Example:
  Assume we have a vocabulary of 5 words:
  `["I", "love", "NLP", "is", "fun"]`
  Then, one-hot encodings would be:
  `I    : [1, 0, 0, 0, 0]`
  `love :  [0, 1, 0, 0, 0]`
  `NLP  : [0, 0, 1, 0, 0]`
  `is   : [0, 0, 0, 1, 0]`
  `fun  : [0, 0, 0, 0, 1]`




In [17]:
from sklearn.preprocessing import OneHotEncoder

In [18]:
encoder = OneHotEncoder(sparse_output=False)

one_hot = encoder.fit_transform(df[['text']])

print(one_hot)

print(encoder.get_feature_names_out())

[[0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]]
['text_food cooks good' 'text_food is food' 'text_food smells good'
 'text_he cooks food']


In [19]:
words_array = np.array(['I', "Love", "NLP", "is", "Fun", "NLP"]).reshape(-1, 1)

encoder = OneHotEncoder()

one_hot = encoder.fit_transform(words_array)

print(one_hot)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6 stored elements and shape (6, 5)>
  Coords	Values
  (0, 1)	1.0
  (1, 2)	1.0
  (2, 3)	1.0
  (3, 4)	1.0
  (4, 0)	1.0
  (5, 3)	1.0


In [20]:
# 1. Tokenize sentences into words
all_words = []

for sentence in df['text']:
    words = sentence.split()
    all_words.extend(words)


# 2. Get unique vocabulary
vocabulary = sorted(set(all_words))

print("\nVocabulary:")
print(vocabulary)


Vocabulary:
['cooks', 'food', 'good', 'he', 'is', 'smells']


In [21]:
# 3. Convert words into DataFrame
word_dataframe = pd.DataFrame(vocabulary, columns=['word'])

print("\nWord DataFrame:")
print(word_dataframe)


Word DataFrame:
     word
0   cooks
1    food
2    good
3      he
4      is
5  smells


In [22]:
# 4. Apply OneHotEncoder on words
encoder = OneHotEncoder(sparse_output=False)

word_one_hot = encoder.fit_transform(word_dataframe[['word']])

print("\nOne Hot Encoding:")
print(word_one_hot)

print("\nFeature Names:")
print(encoder.get_feature_names_out())


One Hot Encoding:
[[1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 1.]]

Feature Names:
['word_cooks' 'word_food' 'word_good' 'word_he' 'word_is' 'word_smells']


In [23]:
word_one_hot.shape

(6, 6)

### limitations of One-Hot Encoding
1. High dimensionality: If you have 10,000 words → 10,000-length vectors
2. No semantics: All vectors are equally distant (no meaning)
3. Memory inefficient: Mostly zeros → sparse and wasteful

### One-Hot Encoding Visualizer

In [24]:
# ==========================================================
# One-Hot Encoding Visualizer
# ==========================================================

!pip -q install gradio

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import gradio as gr

sns.set_style("whitegrid")

def one_hot_visualizer(text):

    # Input
    categories = [x.strip() for x in text.split(",") if x.strip()]

    if len(categories) == 0:
        return None, None

    # Original DataFrame
    df = pd.DataFrame({"Category": categories})

    # One Hot Encoding
    encoded = pd.get_dummies(df, columns=["Category"], dtype=int)

    # Heatmap
    cols = [c for c in encoded.columns if c.startswith("Category_")]

    plt.figure(figsize=(max(6,len(cols)), max(3,len(df)/2)))

    sns.heatmap(
        encoded[cols],
        annot=True,
        cmap="YlGnBu",
        linewidths=1,
        cbar=False,
        fmt="d"
    )

    plt.title("One-Hot Encoding Matrix")
    plt.xlabel("Encoded Features")
    plt.ylabel("Samples")
    plt.tight_layout()

    return encoded, plt.gcf()


demo = gr.Interface(
    fn=one_hot_visualizer,
    inputs=gr.Textbox(
        lines=2,
        label="Enter Categories (comma separated)",
        placeholder="Apple, Banana, Apple, Mango, Banana, Orange"
    ),
    outputs=[
        gr.Dataframe(label="One-Hot Encoded Data"),
        gr.Plot(label="Visualization")
    ],
    title="🎓 One-Hot Encoding (OHE) Visualizer",
    description="""
Type any categorical values separated by commas.

Examples:

Apple, Banana, Apple, Mango

Cat, Dog, Bird, Dog, Cat

Red, Blue, Green, Blue

AI, Web, AI, Cyber, Data Science
"""
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5eadb1aba0c22aa4b4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 2. Bag of Words (BoW)

- Counts how many times each word appears in a document.
- Ignores grammar and word order.
- Simple text classification tasks (e.g., spam detection, topic classification).


In [25]:
# Initialize CountVectorizer (default: Bag-of-Words model)
CV = CountVectorizer(binary=True)

# Fit the vectorizer to the text data and transform it into a sparse matrix representation
bow = CV.fit_transform(df['text'])

# Extract the vocabulary (a dictionary mapping words to their unique indices)
Vocabulary = CV.vocabulary_
# Print the generated vocabulary dictionary
print(Vocabulary)

{'he': 3, 'cooks': 0, 'food': 1, 'good': 2, 'smells': 5, 'is': 4}


In [26]:
# Sorting by index
Vocabulary_sorted = dict(sorted(Vocabulary.items(), key=lambda item: item[1]))
print(Vocabulary_sorted)

{'cooks': 0, 'food': 1, 'good': 2, 'he': 3, 'is': 4, 'smells': 5}


In [27]:
print(bow[0].toarray())
print(bow[1].toarray())
print(bow[2].toarray())
print(bow[3].toarray())

[[1 1 0 1 0 0]]
[[1 1 1 0 0 0]]
[[0 1 1 0 0 1]]
[[0 1 0 0 1 0]]


In [28]:
# Convert to DataFrame for tabular representation
bow_df = pd.DataFrame(bow.toarray(), columns=CV.get_feature_names_out())

# Add original sentences for reference
bow_df.insert(0, "Sentence", df['text'])

# Print the table
print(bow_df.to_string(index=False))

        Sentence  cooks  food  good  he  is  smells
   he cooks food      1     1     0   1   0       0
 food cooks good      1     1     1   0   0       0
food smells good      0     1     1   0   0       1
    food is food      0     1     0   0   1       0


In [29]:
# Transform new text to feature matrix
test_text = ["I went to eat food. The food was cooked good, and smells good"]
test_bow = CV.transform(test_text).toarray()[0]  # Convert to array

In [30]:
test_bow

array([0, 1, 1, 0, 0, 1])

In [31]:
# Prepare tabular output
table = []
for word, index in Vocabulary_sorted.items():
    frequency = test_bow[index] if index < len(test_bow) else 0
                                # Ensure index is within bounds
    table.append([index, word, frequency])

# Create DataFrame for tabular display
df_table = pd.DataFrame(table, columns=["Item Index", "Item Name", "Frequency"])

# Print the table
print(df_table.to_string(index=False))

 Item Index Item Name  Frequency
          0     cooks          0
          1      food          1
          2      good          1
          3        he          0
          4        is          0
          5    smells          1


https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html

In [32]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
# Sample dataset
df = pd.DataFrame({'text': ["he cooks food", "food cooks good", "food smells good",\
                            "food is food"], 'output':[0,1,1,0]})
print(df)

               text  output
0     he cooks food       0
1   food cooks good       1
2  food smells good       1
3      food is food       0


### Bag of Words (BoW) Visualizer

In [33]:
# ==========================================================
# Live Bag of Words (BoW) Visualizer
# ==========================================================

!pip -q install gradio

import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer

sns.set_theme(style="whitegrid")

# ----------------------------------------------------------
# Function
# ----------------------------------------------------------

def bow_demo(text):

    if text.strip() == "":
        return None, None, None

    # Split sentences using new lines
    documents = [doc.strip() for doc in text.split("\n") if doc.strip()]

    vectorizer = CountVectorizer()

    bow = vectorizer.fit_transform(documents)

    bow_df = pd.DataFrame(
        bow.toarray(),
        columns=vectorizer.get_feature_names_out(),
        index=[f"Document {i+1}" for i in range(len(documents))]
    )

    # ---------------- Heatmap ----------------

    plt.figure(figsize=(max(8,len(bow_df.columns)), max(3,len(documents))))

    sns.heatmap(
        bow_df,
        annot=True,
        cmap="YlGnBu",
        linewidths=1,
        fmt="d"
    )

    plt.title("Bag of Words Matrix")
    plt.tight_layout()

    heatmap = plt.gcf()
    plt.close()

    # ---------------- Bar Chart ----------------

    word_counts = bow_df.sum().sort_values(ascending=False)

    plt.figure(figsize=(8,4))

    sns.barplot(
        x=word_counts.values,
        y=word_counts.index,
        palette="viridis"
    )

    plt.title("Total Word Frequency")
    plt.xlabel("Count")
    plt.ylabel("Word")
    plt.tight_layout()

    bar = plt.gcf()
    plt.close()

    return bow_df, heatmap, bar


# ----------------------------------------------------------
# Interface
# ----------------------------------------------------------

examples = [[
"""I love AI
I love Machine Learning
AI is amazing
Machine Learning loves data"""
],
[
"""Pizza Pizza Burger
Burger Fries Pizza
Fries Fries Burger"""
],
[
"""Cat Dog Cat
Dog Bird
Cat Bird Bird"""
]
]

demo = gr.Interface(
    fn=bow_demo,

    inputs=gr.Textbox(
        lines=8,
        label="Enter Documents (One Document Per Line)",
        placeholder="""Example:

I love AI
I love Machine Learning
AI is amazing
Machine Learning loves data"""
    ),

    outputs=[
        gr.Dataframe(label="Bag of Words Matrix"),
        gr.Plot(label="Heatmap"),
        gr.Plot(label="Word Frequency")
    ],

    examples=examples,

    title="📚 Live Bag of Words (BoW) Visualizer",

    description="""
Type multiple documents (one per line).

The application will automatically:

✅ Build the vocabulary

✅ Count the frequency of every word

✅ Create the Bag of Words Matrix

✅ Display a colorful Heatmap

✅ Display Word Frequency
"""
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ba0b8bf68357edcec5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 3. N-Grams
- An n-gram is a contiguous sequence of n items (usually words) from a given text or speech.
  - `A unigram is a single word (n = 1).`
  - `A bigram is a pair of consecutive words (n = 2).`
  - `A trigram is a sequence of three words (n = 3).`
  - And so on...
- N-grams help preserve context and word order compared to Bag of Words.
- Example: "I love natural language processing":
  - `1 (unigram): I, love, natural, language, processing`
  - `2 (bigram): I love, love natural, natural language, language processing`
  - `3 (trigram): I love natural, love natural language, natural language processing`



In [34]:
# Initialize CountVectorizer with bigram (2-gram) model
CV = CountVectorizer(ngram_range=(2, 2), binary=True)

# Fit the vectorizer to the text data and transform it into a sparse matrix representation
bigram_bow = CV.fit_transform(df['text'])

# Extract the vocabulary (a dictionary mapping bigrams to their unique indices)
Vocabulary = CV.vocabulary_
# Print the generated vocabulary dictionary
print(Vocabulary)

# Sorting by index
Vocabulary_sorted = dict(sorted(Vocabulary.items(), key=lambda item: item[1]))
print(Vocabulary_sorted)

{'he cooks': 5, 'cooks food': 0, 'food cooks': 2, 'cooks good': 1, 'food smells': 4, 'smells good': 7, 'food is': 3, 'is food': 6}
{'cooks food': 0, 'cooks good': 1, 'food cooks': 2, 'food is': 3, 'food smells': 4, 'he cooks': 5, 'is food': 6, 'smells good': 7}


In [35]:
# Convert to DataFrame for tabular representation
bigram_bow_df = pd.DataFrame(bigram_bow.toarray(), columns=CV.get_feature_names_out())

# Add original sentences for reference
bigram_bow_df.insert(0, "Sentence", df['text'])

# Print the table
print(bigram_bow_df.to_string(index=False))

        Sentence  cooks food  cooks good  food cooks  food is  food smells  he cooks  is food  smells good
   he cooks food           1           0           0        0            0         1        0            0
 food cooks good           0           1           1        0            0         0        0            0
food smells good           0           0           0        0            1         0        0            1
    food is food           0           0           0        1            0         0        1            0


### Bi-Gram Example 2

In [36]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
# Sample dataset
df = pd.DataFrame({'text': ["he cooks food", "food cooks good", "food smells good",\
                            "he cooks good food", "he cooks bad food"], \
                   'output':[0,1,1,1,0]})
print(df)

                 text  output
0       he cooks food       0
1     food cooks good       1
2    food smells good       1
3  he cooks good food       1
4   he cooks bad food       0


In [37]:
# Initialize CountVectorizer with bigram (2-gram) model
CV = CountVectorizer(ngram_range=(1, 1), binary=True)

# Fit the vectorizer to the text data and transform it into a sparse matrix representation
bigram_bow = CV.fit_transform(df['text'])

# Extract the vocabulary (a dictionary mapping bigrams to their unique indices)
Vocabulary = CV.vocabulary_
# Print the generated vocabulary dictionary
print(Vocabulary)

# Sorting by index
Vocabulary_sorted = dict(sorted(Vocabulary.items(), key=lambda item: item[1]))
print(Vocabulary_sorted)

{'he': 4, 'cooks': 1, 'food': 2, 'good': 3, 'smells': 5, 'bad': 0}
{'bad': 0, 'cooks': 1, 'food': 2, 'good': 3, 'he': 4, 'smells': 5}


In [38]:
# Convert to DataFrame for tabular representation
bigram_bow_df = pd.DataFrame(bigram_bow.toarray(), columns=CV.get_feature_names_out())

# Add original sentences for reference
bigram_bow_df.insert(0, "Sentence", df['text'])

# Print the table
print(bigram_bow_df.to_string(index=False))

          Sentence  bad  cooks  food  good  he  smells
     he cooks food    0      1     1     0   1       0
   food cooks good    0      1     1     1   0       0
  food smells good    0      0     1     1   0       1
he cooks good food    0      1     1     1   1       0
 he cooks bad food    1      1     1     0   1       0


### Tri-Gram

In [39]:
# Initialize CountVectorizer with bigram (3-gram) model
CV = CountVectorizer(ngram_range=(3, 3), binary=True)
# Fit the vectorizer to the text data and transform it into a sparse matrix representation

trigram_bow = CV.fit_transform(df['text'])
# Extract the vocabulary (a dictionary mapping bigrams to their unique indices)
Vocabulary = CV.vocabulary_
# Print the generated vocabulary dictionary
print(Vocabulary)

# Sorting by index
Vocabulary_sorted = dict(sorted(Vocabulary.items(), key=lambda item: item[1]))
print(Vocabulary_sorted)

{'he cooks food': 5, 'food cooks good': 2, 'food smells good': 3, 'he cooks good': 6, 'cooks good food': 1, 'he cooks bad': 4, 'cooks bad food': 0}
{'cooks bad food': 0, 'cooks good food': 1, 'food cooks good': 2, 'food smells good': 3, 'he cooks bad': 4, 'he cooks food': 5, 'he cooks good': 6}


https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html

### N-Gram Visualizer

In [3]:
# ==========================================================
# Live N-Gram Visualizer
# ==========================================================

!pip -q install gradio

import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer

sns.set_theme(style="whitegrid")


# ----------------------------------------------------------
# Function
# ----------------------------------------------------------

def ngram_demo(text, n):

    text = text.strip()

    if text == "":
        return None, None, None, None

    # Generate n-grams
    vectorizer = CountVectorizer(ngram_range=(n, n))

    X = vectorizer.fit_transform([text])

    vocabulary = vectorizer.get_feature_names_out()

    # DataFrame
    bow_df = pd.DataFrame(
        X.toarray(),
        columns=vocabulary,
        index=["Sentence"]
    )

    # List of n-grams
    ngram_df = pd.DataFrame({
        f"{n}-Gram": vocabulary
    })

    # ---------------------------
    # Frequency Plot
    # ---------------------------

    frequency = bow_df.iloc[0]

    plt.figure(figsize=(max(8, len(vocabulary)*0.8),4))

    sns.barplot(
        x=frequency.values,
        y=frequency.index,
        palette="viridis"
    )

    plt.title(f"{n}-Gram Frequency")
    plt.xlabel("Count")
    plt.ylabel(f"{n}-Gram")
    plt.tight_layout()

    bar = plt.gcf()
    plt.close()

    # ---------------------------
    # Heatmap
    # ---------------------------

    plt.figure(figsize=(max(8, len(vocabulary)*0.8),2.5))

    sns.heatmap(
        bow_df,
        annot=True,
        cmap="YlGnBu",
        linewidths=1,
        fmt="d",
        cbar=False
    )

    plt.title(f"{n}-Gram Matrix")
    plt.tight_layout()

    heat = plt.gcf()
    plt.close()

    return ngram_df, bow_df, heat, bar


# ----------------------------------------------------------
# Examples
# ----------------------------------------------------------

examples = [

["Artificial Intelligence is changing the world",2],

["Machine Learning makes AI smarter every day",3],

["The quick brown fox jumps over the lazy dog",2],

["Natural Language Processing is fun to learn",4]

]


# ----------------------------------------------------------
# Interface
# ----------------------------------------------------------

demo = gr.Interface(

    fn=ngram_demo,

    inputs=[

        gr.Textbox(
            lines=4,
            label="Enter a Sentence",
            placeholder="Example: Artificial Intelligence is changing the world"
        ),

        gr.Slider(
            minimum=1,
            maximum=4,
            value=2,
            step=1,
            label="Choose n"
        )

    ],

    outputs=[

        gr.Dataframe(label="Generated N-Grams"),

        gr.Dataframe(label="N-Gram Count Matrix"),

        gr.Plot(label="Heatmap"),

        gr.Plot(label="Frequency Chart")

    ],

    examples=examples,

    title="📚 Live N-Gram Visualizer",

    description="""
🎯 Enter any sentence and select **n**.

The app will instantly show:

✅ Generated n-grams

✅ CountVectorizer matrix

✅ Heatmap

✅ Frequency chart

Try changing n from 1 → 2 → 3 → 4 to see how the generated phrases change.
"""

)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0839bd2db0c6725dae.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 4. TF-IDF
  - TF-IDF stands for:
    - TF – Term Frequency: how often a word appears in a document.
    - IDF – Inverse Document Frequency: how unique or rare a word is across all documents.
  - Together, TF-IDF measures how important a word is to a specific document in a collection


In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Sample dataset
df = pd.DataFrame({'text': ["he cooks food", "food cooks good", "food smells good",
                            "he cooks good food", "he cooks bad food"],
                   'output':[0,1,1,1,0]})

# Initialize TfidfVectorizer with unigram (1-gram) model
TFIDF = TfidfVectorizer()

# Fit the vectorizer to the text data and transform it into a TF-IDF matrix
tfidf_matrix = TFIDF.fit_transform(df['text'])

# Extract the vocabulary (a dictionary mapping terms to their unique indices)
Vocabulary = TFIDF.vocabulary_

# Print the generated vocabulary dictionary
print(Vocabulary)

# Sorting vocabulary by index
Vocabulary_sorted = dict(sorted(Vocabulary.items(), key=lambda item: item[1]))
print(Vocabulary_sorted)

{'he': 4, 'cooks': 1, 'food': 2, 'good': 3, 'smells': 5, 'bad': 0}
{'bad': 0, 'cooks': 1, 'food': 2, 'good': 3, 'he': 4, 'smells': 5}


In [42]:
# Convert to DataFrame for tabular representation
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=TFIDF.get_feature_names_out())

# Add original sentences for reference
tfidf_df.insert(0, "Sentence", df['text'])

# Print the table
print(tfidf_df.to_string(index=False))

          Sentence      bad    cooks     food     good       he   smells
     he cooks food 0.000000 0.565373 0.478189 0.000000 0.672078 0.000000
   food cooks good 0.000000 0.565373 0.478189 0.672078 0.000000 0.000000
  food smells good 0.000000 0.000000 0.368117 0.517376 0.000000 0.772536
he cooks good food 0.000000 0.469244 0.396883 0.557806 0.557806 0.000000
 he cooks bad food 0.708353 0.399074 0.337534 0.000000 0.474392 0.000000


In [43]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import pandas as pd
import numpy as np

# Sample dataset
df = pd.DataFrame({'text': ["he cooks food", "food cooks good", "food smells good",
                            "he cooks good food", "he cooks bad food"],
                   'output':[0,1,1,1,0]})

# Step 1: Compute Term Frequency (TF)
CV = CountVectorizer(ngram_range=(1, 1))  # Using Unigram Model
bow_matrix = CV.fit_transform(df['text'])  # Bag of Words Count
feature_names = CV.get_feature_names_out()

# Convert BoW to DataFrame
tf_raw_df = pd.DataFrame(bow_matrix.toarray(), columns=feature_names)

# Normalize TF by dividing by total words in each sentence
tf_normalized_df = tf_raw_df.div(tf_raw_df.sum(axis=1), axis=0)
tf_normalized_df = tf_normalized_df.round(2)
# Add original sentences for reference
tf_normalized_df.insert(0, "Sentence", df['text'])

print("\n🔹 Term Frequency (TF) Table (Normalized by Total Words):\n")
print(tf_normalized_df.to_string(index=False))


# Step 2: Compute Inverse Document Frequency (IDF)
TFIDF = TfidfVectorizer()
TFIDF.fit(df['text'])  # Fit the model to extract IDF values
# Extract IDF values
idf_values = TFIDF.idf_
idf_df = pd.DataFrame({'Word': TFIDF.get_feature_names_out(), 'IDF': np.round(idf_values, 2)})

print("\n🔹 Inverse Document Frequency (IDF) Table:\n")
print(idf_df.to_string(index=False))

# Step 3: Compute TF-IDF
tfidf_matrix = TFIDF.transform(df['text'])

# Convert TF-IDF matrix to DataFrame
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=TFIDF.get_feature_names_out())

# Round the TF-IDF values to two decimal places
tfidf_df = tfidf_df.round(2)

# Add original sentences for reference
tfidf_df.insert(0, "Sentence", df['text'])

print("\n🔹 TF-IDF Table:\n")
print(tfidf_df.to_string(index=False))


🔹 Term Frequency (TF) Table (Normalized by Total Words):

          Sentence  bad  cooks  food  good   he  smells
     he cooks food 0.00   0.33  0.33  0.00 0.33    0.00
   food cooks good 0.00   0.33  0.33  0.33 0.00    0.00
  food smells good 0.00   0.00  0.33  0.33 0.00    0.33
he cooks good food 0.00   0.25  0.25  0.25 0.25    0.00
 he cooks bad food 0.25   0.25  0.25  0.00 0.25    0.00

🔹 Inverse Document Frequency (IDF) Table:

  Word  IDF
   bad 2.10
 cooks 1.18
  food 1.00
  good 1.41
    he 1.41
smells 2.10

🔹 TF-IDF Table:

          Sentence  bad  cooks  food  good   he  smells
     he cooks food 0.00   0.57  0.48  0.00 0.67    0.00
   food cooks good 0.00   0.57  0.48  0.67 0.00    0.00
  food smells good 0.00   0.00  0.37  0.52 0.00    0.77
he cooks good food 0.00   0.47  0.40  0.56 0.56    0.00
 he cooks bad food 0.71   0.40  0.34  0.00 0.47    0.00


In [44]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np

# Sample dataset
df = pd.DataFrame({'text': ["he cooks food", "food cooks good", "food smells good",
                            "he cooks good food", "he cooks bad food"],
                   'output':[0,1,1,1,0]})
print(df)

                 text  output
0       he cooks food       0
1     food cooks good       1
2    food smells good       1
3  he cooks good food       1
4   he cooks bad food       0


In [45]:
# Step 1: Compute Term Frequency (TF) (normalized)
CV = TfidfVectorizer()  # Calculate Only Term Frequency without IDF

tf_matrix = CV.fit_transform(df['text']).toarray()  # Get raw TF counts

feature_names = CV.get_feature_names_out()

# Normalize TF by dividing by total words in each document
tf_normalized = tf_matrix / tf_matrix.sum(axis=1, keepdims=True)
tf_normalized = tf_normalized.round(2)

# Convert TF matrix to DataFrame
tf_df = pd.DataFrame(tf_normalized, columns=feature_names)
tf_df.insert(0, "Sentence", df['text'])  # Add original text for reference

print("\n🔹 Term Frequency (TF) Table (Normalized by Total Words):\n")

print(tf_df.to_string(index=False))


🔹 Term Frequency (TF) Table (Normalized by Total Words):

          Sentence  bad  cooks  food  good   he  smells
     he cooks food 0.00   0.33  0.28  0.00 0.39    0.00
   food cooks good 0.00   0.33  0.28  0.39 0.00    0.00
  food smells good 0.00   0.00  0.22  0.31 0.00    0.47
he cooks good food 0.00   0.24  0.20  0.28 0.28    0.00
 he cooks bad food 0.37   0.21  0.18  0.00 0.25    0.00


In [46]:
# Step 2: Compute Inverse Document Frequency (IDF)
N = len(df)  # Total number of documents
df_counts = np.sum(tf_matrix > 0, axis=0)  # Count documents containing each word
idf_manual = np.log(N / df_counts) + 1  # Apply IDF formula using natural log
# Compute IDF using TfidfVectorizer
TFIDF = TfidfVectorizer()
TFIDF.fit(df['text'])
idf_sklearn = TFIDF.idf_  # Extract IDF values from sklearn
# Create IDF comparison DataFrame
idf_df = pd.DataFrame({'Word': feature_names, 'IDF (Manual)': np.round(idf_manual, 5), \
                       'IDF (Sklearn)': np.round(idf_sklearn, 5)})
print("\n🔹 Inverse Document Frequency (IDF) Table (Manual vs Sklearn):\n")
print(idf_df.to_string(index=False))


🔹 Inverse Document Frequency (IDF) Table (Manual vs Sklearn):

  Word  IDF (Manual)  IDF (Sklearn)
   bad       2.60944        2.09861
 cooks       1.22314        1.18232
  food       1.00000        1.00000
  good       1.51083        1.40547
    he       1.51083        1.40547
smells       2.60944        2.09861


In [47]:
# Step 3: Compute TF-IDF using both methods
tfidf_manual = tf_normalized * idf_manual  # TF (Normalized) * IDF (Manual)
tfidf_sklearn = TFIDF.transform(df['text']).toarray()  # Sklearn's TF-IDF
# Convert TF-IDF matrices to DataFrames
tfidf_manual_df = pd.DataFrame(tfidf_manual, columns=feature_names)
tfidf_manual_df.insert(0, "Sentence", df['text'])
tfidf_sklearn_df = pd.DataFrame(tfidf_sklearn, columns=feature_names)
tfidf_sklearn_df.insert(0, "Sentence", df['text'])
print("\n🔹 TF-IDF Table (Manual Calculation):\n")
print(tfidf_manual_df.to_string(index=False))
print("\n🔹 TF-IDF Table (Using Sklearn):\n")
print(tfidf_sklearn_df.to_string(index=False))


🔹 TF-IDF Table (Manual Calculation):

          Sentence      bad    cooks  food     good       he   smells
     he cooks food 0.000000 0.403637  0.28 0.000000 0.589222 0.000000
   food cooks good 0.000000 0.403637  0.28 0.589222 0.000000 0.000000
  food smells good 0.000000 0.000000  0.22 0.468356 0.000000 1.226436
he cooks good food 0.000000 0.293554  0.20 0.423031 0.423031 0.000000
 he cooks bad food 0.965492 0.256860  0.18 0.000000 0.377706 0.000000

🔹 TF-IDF Table (Using Sklearn):

          Sentence      bad    cooks     food     good       he   smells
     he cooks food 0.000000 0.565373 0.478189 0.000000 0.672078 0.000000
   food cooks good 0.000000 0.565373 0.478189 0.672078 0.000000 0.000000
  food smells good 0.000000 0.000000 0.368117 0.517376 0.000000 0.772536
he cooks good food 0.000000 0.469244 0.396883 0.557806 0.557806 0.000000
 he cooks bad food 0.708353 0.399074 0.337534 0.000000 0.474392 0.000000


#### Limitations:
- No semantic meaning: “love” and “like” are unrelated numerically
- Still sparse vectors: May be inefficient for very large vocabularies
- Ignores word order: Can’t detect phrases or grammar


#### Summary:
- TF How often a word occurs in a document
- IDF How rare the word is across all docs
- TF-IDF Importance of a word in a document


### TF-IDF Visualizer

In [4]:
# ==========================================================
# Live TF-IDF Visualizer
# ==========================================================

!pip -q install gradio

import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer

sns.set_theme(style="whitegrid")


# ==========================================================
# Function
# ==========================================================

def tfidf_demo(text):

    if text.strip() == "":
        return None, None, None, None, None, None, None

    # ------------------------------------------
    # Documents
    # ------------------------------------------

    docs = [d.strip() for d in text.split("\n") if d.strip()]

    vectorizer = CountVectorizer()

    counts = vectorizer.fit_transform(docs)

    words = vectorizer.get_feature_names_out()

    count_df = pd.DataFrame(
        counts.toarray(),
        columns=words,
        index=[f"Doc {i+1}" for i in range(len(docs))
    ])

    # ------------------------------------------
    # TF
    # ------------------------------------------

    tf_df = count_df.div(count_df.sum(axis=1), axis=0).round(3)

    # ------------------------------------------
    # IDF
    # ------------------------------------------

    transformer = TfidfTransformer(norm=None)

    transformer.fit(counts)

    idf_df = pd.DataFrame({

        "Word": words,

        "IDF": transformer.idf_.round(3)

    }).sort_values("IDF")

    # ------------------------------------------
    # TF-IDF
    # ------------------------------------------

    tfidf = transformer.transform(counts)

    tfidf_df = pd.DataFrame(
        tfidf.toarray().round(3),
        columns=words,
        index=[f"Doc {i+1}" for i in range(len(docs))]
    )

    # ------------------------------------------
    # TF Heatmap
    # ------------------------------------------

    plt.figure(figsize=(max(8,len(words)), max(3,len(docs))))

    sns.heatmap(
        tf_df,
        annot=True,
        cmap="Greens",
        linewidths=1
    )

    plt.title("TF Matrix")

    plt.tight_layout()

    tf_plot = plt.gcf()

    plt.close()

    # ------------------------------------------
    # IDF Plot
    # ------------------------------------------

    plt.figure(figsize=(8,4))

    sns.barplot(
        data=idf_df,
        x="IDF",
        y="Word",
        palette="viridis"
    )

    plt.title("Inverse Document Frequency")

    plt.tight_layout()

    idf_plot = plt.gcf()

    plt.close()

    # ------------------------------------------
    # TF-IDF Heatmap
    # ------------------------------------------

    plt.figure(figsize=(max(8,len(words)), max(3,len(docs))))

    sns.heatmap(
        tfidf_df,
        annot=True,
        cmap="YlOrRd",
        linewidths=1
    )

    plt.title("TF-IDF Matrix")

    plt.tight_layout()

    tfidf_plot = plt.gcf()

    plt.close()

    return (
        count_df,
        tf_df,
        idf_df,
        tfidf_df,
        tf_plot,
        idf_plot,
        tfidf_plot
    )


# ==========================================================
# Examples
# ==========================================================

examples = [[
"""movie movie movie action hero
movie comedy hero
movie action adventure
movie masterpiece"""
],

[
"""apple apple apple fruit
apple banana fruit
banana orange fruit
dragonfruit"""
],

[
"""AI AI AI machine learning
AI deep learning
machine learning
artificial intelligence"""
]
]


# ==========================================================
# Interface
# ==========================================================

demo = gr.Interface(

    fn=tfidf_demo,

    inputs=gr.Textbox(

        lines=8,

        label="Enter Documents (One Document Per Line)",

        placeholder="""movie movie movie action hero
movie comedy hero
movie action adventure
movie masterpiece"""

    ),

    outputs=[

        gr.Dataframe(label="1️⃣ Count Matrix"),

        gr.Dataframe(label="2️⃣ TF Matrix"),

        gr.Dataframe(label="3️⃣ IDF Values"),

        gr.Dataframe(label="4️⃣ TF-IDF Matrix"),

        gr.Plot(label="TF Heatmap"),

        gr.Plot(label="IDF Plot"),

        gr.Plot(label="TF-IDF Heatmap")

    ],

    examples=examples,

    title="📚 Live TF-IDF Visualizer",

    description="""
Type one document per line.

The application automatically calculates:

✅ Count Matrix

✅ TF (Term Frequency)

✅ IDF (Inverse Document Frequency)

✅ TF-IDF Matrix

and visualizes every step.
"""

)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e37765e418376f625.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Word Embeddings:
* Word embeddings are dense vector representations of words in a continuous vector space,
where similar words are mapped to similar vectors.
* vector("king") - vector("man") + vector("woman") ≈ vector("queen")
* Word embedding is a way to turn words into numbers, so a computer can understand them
— but not just any numbers.
* It represents each word as a list of numbers (a vector) where:
  - Similar words (like "king" and "queen") get similar numbers
  - Different words (like "apple" and "car") get very different numbers
* So the computer can figure out which words are related and how closely.
 - `king : [0.2, 0.5, -0.3]`
 - `queen :[0.2, 0.6, -0.3]`
 - `apple : [-0.7, 0.1, 0.4]`
 - `king` and `queen` have similar vectors → so the computer knows they are related.


## Why Do We Need Word Embeddings?
* Problem with Traditional Approaches:
  - One-hot vectors are huge & sparse
  - No meaning or similarity captured
  - Cannot generalize across contexts
* Word Embeddings Solve:
  - Embeddings are dense & compact
  - Embeddings group similar meanings together
  - Embeddings help capture word usage patterns
* Each word is represented as a vector of real numbers (e.g., 100–300 dimensions), trained so
that words used in similar contexts have similar vectors.
* Common Word Embedding Models:
  - `Word2Vec: Predicts a word from its context (or vice versa)`
  - `GloVe: Builds word vectors using matrix factorization of word co-occurrence`
  - `FastText: Includes subword information (good for misspellings and rare words)`
  - `ELMo, BERT: Contextual embeddings (meaning changes based on sentence context)`

In [ ]:
import numpy as np
import pandas as pd

In [5]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 21.7 MB/s eta 0:00:00


In [6]:
import gensim as gn, os

In [7]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [8]:
from nltk import sent_tokenize

In [9]:
# Sample sentences

sentences = [
    "king is a strong man",
    "queen is a wise woman",
    "boy is a young man",
    "girl is a young woman",
    "prince is a young king",
    "princess is a young queen",
    "man is strong",
    "woman is pretty",
]

for s in sentences:
    print(s)

king is a strong man
queen is a wise woman
boy is a young man
girl is a young woman
prince is a young king
princess is a young queen
man is strong
woman is pretty


In [10]:
sentence = "the cat sat on the mat".split()

window_size = 2

print("Sentence:\n", sentence)
print()

for i, word in enumerate(sentence):

    start = max(0, i-window_size)
    end = min(len(sentence), i+window_size+1)

    context = []

    for j in range(start, end):
        if j != i:
            context.append(sentence[j])

    print(f"Target : {word}")
    print(f"Context: {context}")
    print("-"*40)

Sentence:
 ['the', 'cat', 'sat', 'on', 'the', 'mat']

Target : the
Context: ['cat', 'sat']
----------------------------------------
Target : cat
Context: ['the', 'sat', 'on']
----------------------------------------
Target : sat
Context: ['the', 'cat', 'on', 'the']
----------------------------------------
Target : on
Context: ['cat', 'sat', 'the', 'mat']
----------------------------------------
Target : the
Context: ['sat', 'on', 'mat']
----------------------------------------
Target : mat
Context: ['on', 'the']
----------------------------------------


In [11]:
from gensim.models import Word2Vec

sentences = [
    "king is a strong man".split(),
    "queen is a wise woman".split(),
    "boy is a young man".split(),
    "girl is a young woman".split(),
    "prince is a young king".split(),
    "princess is a young queen".split(),
    "man is strong".split(),
    "woman is pretty".split(),
]

model = Word2Vec(
    sentences,
    vector_size=20,
    window=2,
    min_count=1,
    workers=1,
    epochs=300
)

print("Vocabulary\n")
print(model.wv.index_to_key)

print("\nEmbedding of KING\n")
print(model.wv["king"])

Vocabulary

['is', 'a', 'young', 'woman', 'man', 'queen', 'strong', 'king', 'pretty', 'princess', 'prince', 'girl', 'boy', 'wise']

Embedding of KING

[ 3.77665795e-02 -3.21108103e-02 -4.25402075e-02 -1.39783800e-03
 -1.53229088e-02  3.80434170e-02  3.22724804e-02  2.33408762e-04
  1.75531791e-03  1.08302925e-02  4.11124490e-02 -4.95760702e-02
 -2.55464856e-05  1.48676913e-02 -3.49838287e-03  4.23149131e-02
  5.36939912e-02  3.09187230e-02 -8.67055077e-03  3.66217084e-02]


In [12]:
print("Words similar to KING\n")

for word, score in model.wv.most_similar("king"):

    print(f"{word:12} {score:.3f}")

print()

print("Similarity")

print("King vs Queen :",
      model.wv.similarity("king","queen"))

print("King vs Princess :",
      model.wv.similarity("king","princess"))

print("King vs Woman :",
      model.wv.similarity("king","woman"))

Words similar to KING

wise         0.452
strong       0.338
girl         0.184
prince       0.129
queen        0.101
is           0.047
woman        0.009
man          -0.038
princess     -0.056
pretty       -0.076

Similarity
King vs Queen : 0.10112743
King vs Princess : -0.05577344
King vs Woman : 0.008830659


### FRONT END

In [14]:
import gradio as gr
import pandas as pd
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# -----------------------------
# Sample Corpus
# -----------------------------

sample_text = """king is a strong man
queen is a wise woman
boy is a young man
girl is a young woman
prince is a young king
princess is a young queen
man is strong
woman is pretty
king queen prince princess
man woman boy girl"""

model = None


# ---------------------------------
# Create Context Pairs
# ---------------------------------

def generate_context(text, window):

    sentences = [line.lower().split() for line in text.strip().split("\n")]

    rows = []

    for sentence in sentences:

        for i, target in enumerate(sentence):

            start = max(0, i-window)
            end = min(len(sentence), i+window+1)

            context = []

            for j in range(start,end):
                if j!=i:
                    context.append(sentence[j])

            rows.append([target,", ".join(context)])

    return pd.DataFrame(rows,columns=["Target Word","Context Words"])


# ---------------------------------
# Train Word2Vec
# ---------------------------------

def train_model(text, vector_size, window, epochs):

    global model

    sentences = [line.lower().split() for line in text.strip().split("\n")]

    model = Word2Vec(
        sentences,
        vector_size=vector_size,
        window=window,
        min_count=1,
        workers=1,
        epochs=epochs
    )

    vocab = model.wv.index_to_key

    info = f"""
Vocabulary Size : {len(vocab)}

Vocabulary

{", ".join(vocab)}

Embedding Dimension : {vector_size}

Training Completed Successfully
"""

    return info


# ---------------------------------
# Similar Words
# ---------------------------------

def similar_words(word):

    global model

    if model is None:
        return pd.DataFrame()

    if word not in model.wv:
        return pd.DataFrame()

    sim = model.wv.most_similar(word)

    return pd.DataFrame(sim,columns=["Word","Similarity"])


# ---------------------------------
# Show Vector
# ---------------------------------

def show_vector(word):

    global model

    if model is None:
        return "Train model first."

    if word not in model.wv:
        return "Word not found."

    vector = model.wv[word]

    return str(vector)


# ---------------------------------
# PCA Plot
# ---------------------------------

# def plot_embeddings():

#     global model

#     if model is None:
#         return None

#     words = model.wv.index_to_key

#     vectors = [model.wv[w] for w in words]

#     pca = PCA(n_components=2)

#     points = pca.fit_transform(vectors)

#     fig = plt.figure(figsize=(8,6))

#     for i,word in enumerate(words):

#         plt.scatter(points[i,0],points[i,1])

#         plt.text(points[i,0]+0.02,
#                  points[i,1]+0.02,
#                  word,
#                  fontsize=10)

#     plt.title("Word2Vec Embeddings (PCA)")
#     plt.grid(True)

#     return fig

from mpl_toolkits.mplot3d import Axes3D

# ---------------------------------
# 3D PCA Plot
# ---------------------------------

def plot_embeddings():

    global model

    if model is None:
        return None

    words = model.wv.index_to_key
    vectors = [model.wv[w] for w in words]

    # Reduce to 3 dimensions
    pca = PCA(n_components=3)
    points = pca.fit_transform(vectors)

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    for i, word in enumerate(words):
        ax.scatter(
            points[i, 0],
            points[i, 1],
            points[i, 2],
            s=50
        )

        ax.text(
            points[i, 0],
            points[i, 1],
            points[i, 2],
            word,
            fontsize=10
        )

    ax.set_title("Word2Vec Embeddings (3D PCA)")
    ax.set_xlabel("Principal Component 1")
    ax.set_ylabel("Principal Component 2")
    ax.set_zlabel("Principal Component 3")

    return fig


# ---------------------------------
# GUI
# ---------------------------------

with gr.Blocks(theme=gr.themes.Soft(),
               title="Word2Vec Learning Lab") as demo:

    gr.Markdown(
    """
# 🧠 Word2Vec Interactive Learning Lab

Learn Word2Vec step-by-step.

### Workflow

1. Enter a corpus
2. Observe Context Windows
3. Train Word2Vec
4. Find Similar Words
5. Inspect Word Vectors
6. Visualize Embeddings
""")

    corpus = gr.Textbox(
        value=sample_text,
        lines=12,
        label="Corpus"
    )

    with gr.Tab("Step 1 : Context Window"):

        window_slider = gr.Slider(
            1,
            4,
            value=2,
            step=1,
            label="Window Size"
        )

        context_btn = gr.Button("Generate Training Pairs")

        context_table = gr.Dataframe()

        context_btn.click(
            generate_context,
            [corpus,window_slider],
            context_table
        )

    with gr.Tab("Step 2 : Train Model"):

        vector_slider = gr.Slider(
            10,
            100,
            value=20,
            step=10,
            label="Vector Size"
        )

        epoch_slider = gr.Slider(
            50,
            500,
            value=200,
            step=50,
            label="Epochs"
        )

        train_btn = gr.Button("Train Word2Vec")

        training_info = gr.Textbox(lines=10)

        train_btn.click(
            train_model,
            [corpus,
             vector_slider,
             window_slider,
             epoch_slider],
            training_info
        )

    with gr.Tab("Step 3 : Explore Words"):

        word_box = gr.Textbox(label="Enter Word")

        similar_btn = gr.Button("Find Similar Words")

        similar_table = gr.Dataframe()

        vector_btn = gr.Button("Show Word Vector")

        vector_output = gr.Textbox(lines=8)

        similar_btn.click(
            similar_words,
            word_box,
            similar_table
        )

        vector_btn.click(
            show_vector,
            word_box,
            vector_output
        )

    with gr.Tab("Step 4 : Visualization"):

        plot_btn = gr.Button("Generate PCA Plot")

        plot = gr.Plot()

        plot_btn.click(
            plot_embeddings,
            outputs=plot
        )

demo.launch()

/tmp/ipykernel_1052/3977938451.py:213: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(),


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://40aa1dd9e54a9c3240.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
os.makedirs("data", exist_ok=True)

In [ ]:
!unzip /content/archive.zip -d /content/data

Archive:  /content/archive.zip
  inflating: /content/data/001ssb.txt  
  inflating: /content/data/002ssb.txt  
  inflating: /content/data/003ssb.txt  
  inflating: /content/data/004ssb.txt  
  inflating: /content/data/005ssb.txt  


In [ ]:
list_data = []

In [ ]:
for file in os.listdir("data"):
  print(file)

  with open(os.path.join("data", file), encoding='utf-8', errors='ignore') as f:
    my_data = f.read()

  text = sent_tokenize(my_data)

  for i in text:
    list_data.append(simple_preprocess(i))

003ssb.txt
004ssb.txt
005ssb.txt
002ssb.txt
001ssb.txt


In [ ]:
len(list_data)

158874

In [ ]:
my_model = gn.models.Word2Vec(
    window=10,
    vector_size=100
)

In [ ]:
my_model.build_vocab(list_data)

In [ ]:
my_model.train(list_data, total_examples=my_model.corpus_count, epochs=my_model.epochs)

(6485449, 8625265)

In [ ]:
my_model.wv['king']

array([ 3.8878284 , -0.49143195, -0.59478503, -1.6962724 , -2.8627334 ,
        2.352737  ,  1.6661154 , -0.6060034 ,  0.1453002 ,  2.3781567 ,
       -0.36363697,  0.11056992,  0.43432024,  0.4374635 , -1.1199006 ,
       -2.398084  ,  2.501843  , -2.4066594 , -2.8360875 , -2.3217368 ,
        3.4493144 ,  2.4921265 ,  0.5960854 , -2.450634  ,  0.62053615,
       -0.7986131 , -2.987835  , -0.31277573,  0.66741633,  2.054903  ,
        0.07465312,  1.4429355 , -1.8824944 , -0.87976485,  1.9738601 ,
        0.6300053 ,  1.1415986 ,  0.68980205, -1.7575525 , -0.32072258,
       -2.8930674 , -0.9597652 ,  2.044995  , -0.29497018,  0.14067908,
       -1.9542838 ,  1.0898236 ,  1.1079109 ,  2.6419604 , -0.6650989 ,
       -1.0716685 , -0.21204522,  1.7262665 , -1.8822371 , -0.56582856,
       -0.44558272,  0.03395495,  0.9837842 ,  0.08908229, -0.16751325,
        1.4004788 , -0.30295974, -0.20621973,  1.7163489 ,  0.63329285,
        1.1106207 , -0.7822558 ,  1.2371043 , -1.67224   , -0.97

In [ ]:
my_model.wv.get_normed_vectors().shape

(11975, 100)

In [ ]:
y = my_model.wv.index_to_key

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca = PCA(n_components=3)

In [ ]:
x = pca.fit_transform(my_model.wv.get_normed_vectors())

In [ ]:
x.shape

(11975, 3)

In [ ]:
import plotly.express as px

In [ ]:
px.scatter_3d(x[1000:1010], x=0, y=1, z=2, color=y[1000:1010])